# 3D multichannel segmentation

This notebook demonstrates a complete workflow for segmenting nuclei, cytoplasm, and intracellular structures from **3D microscopy data**, and for quantifying structures at both the object and cell level. The pipeline is modular and extensible, allowing different segmentation strategies depending on the channel and biological question.

The general pipeline includes:

1. Loading and preprocessing volumetric data

    - Intensity normalization
    - Optional downsampling and smoothing
2. Segmentation of cellular compartments

    - Nuclei (using deep learning models: StarDist or Cellpose)

    - Cytoplasm (via intensity- or membrane-based watershed)

    - Detection of intracellular structures

        -AICS segmentation workflows (e.g., dot/filament filters)
3. (optional) Post-processing (object removal, smoothing)

4. Quantification of structures per cell

    - Mapping detected objects to their parent cell

    - Extracting per-object and per-cell features (count, volume, intensity)

    - Exporting results to CSV for downstream analysis

5. Quality control

    - Visual overlays (e.g., maximum intensity projections, per-cell structure maps)
    
The goal is to link each detected intracellular structure to its cell of origin, enabling biologically meaningful, cell-level measurements.

## Load a 3D image 

You need to specify your input folder, how many images to segment, if you want to dowsample, normalise per slice, and the map your channel into a directory:

    - "nucleus": 0 → channel index 0 contains the nuclear stain.

    - "cytoplasm": 2 → channel index 2 contains cytoplasmic signal.

    - "intracellular": 1 → channel index 1 contains the organelle/structure of interest.
    
This mapping depends on the acquisition setup and might change between datasets — check the raw image metadata if in doubt (e.g. via opening a representative example image in ImageJ)

In [ ]:
## Libraries
## Load packages
# import glob, os
import skimage
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import os
import tifffile

# --------------------
# INPUT SETTINGS
# --------------------

input_folder: "/home/ucbtvsi/Image-Analysis-Summer-Project/raw/images"      # Directory containing TIFF image frames
output_folder: "/home/ucbtvsi/Image-Analysis-Summer-Project/output"         # Directory where all outputs will be saved

# --------------------
# --------------------

# 1. load images as multichannel dictionary

# add a dictionary for your images, stating which channel corresponds to which of the structures
channel_map = {
    "nucleus": 0,
    "cytoplasm": 2,
    "intracellular": 1
}

# enter voxel size of your images (obtained from the metadata)
voxel_size_um = [1, 0.48, 0.48 ] # Z, Y, X; from metadata

# loading images from the input folder directory
files = sorted(input_folder.glob("*.tif"))

if not files:
    raise FileNotFoundError("No .tif files found in the folder.")

all_volumes = []

for f in files:
    img = tifffile.imread(f)  # could be (Z,Y,X,C) or (C,Z,Y,X)

    # this is to handle errors
    if img.ndim != 4:
        raise ValueError(f"Unexpected image shape: {img.shape}. Expected 4D (Z,Y,X,C) or (C,Z,Y,X).")
    
    # identify channels axis and move it to last
    if img.shape[-1] <= 10:
            zyx_channels = img  # already in (Z,Y,X,C)
    else:
        # guess channel axis (the one with size < 10)
        channel_axis = np.argmin(img.shape)
        if img.shape[channel_axis] < 10:
                zyx_channels = np.moveaxis(img, channel_axis, -1) # moving the channel axis to the last position
        else:
            raise ValueError(f"Cannot determine channel axis for shape {img.shape}")

    # map channels to names
    channels_dict = {}
    for name, idx in channel_map.items():
        if idx >= zyx_channels.shape[-1]:
            raise IndexError(f"Channel index {idx} for '{name}' out of range in image {f.name}")
        channels_dict[name] = zyx_channels[..., idx]

    # append structured entry with filename and channels (per image)
    all_volumes.append({
        "filename": f.stem, 
        "channels": channels_dict
    })

# preview loaded images: first image [0] as example
print(all_volumes[0]["filename"])       # e.g., "sample01.tif"
print(all_volumes[0]["channels"].keys()) # what channels are included in the dictionary? should be: dict_keys(['nucleus', 'cytoplasm', 'intracellular'])
print(all_volumes[0]["channels"]["nucleus"].shape)  # (Z, Y, X)

## Quick check: middle slice for each channel

We can display the **middle Z-slice** of each channel from the same (first) image of the folder. You can view different images by editing Volume_number.

This allows you to quickly confirm:

    - Channel order (e.g., nucleus, cytoplasm, intracellular)
    - The objects in the different channels should overlap, with nucleus and intracellular objects within the cytoplasm area.
    - That the data is loaded correctly and matches expectations
    
If the signal looks wrong (e.g., nucleus is empty but cytoplasm is bright), check the channel_map definition above.

In [ ]:
### Which volume to view?
Volume_number = 0

# take the first loaded volume
vol = all_volumes[Volume_number]["channels"]

# find middle slice
z_mid = next(iter(vol.values())).shape[0] // 2  

# plot all channels in a grid
n_channels = len(vol)
fig, axes = plt.subplots(1, n_channels, figsize=(5 * n_channels, 5))

for ax, (name, data) in zip(axes, vol.items()):
    ax.imshow(data[z_mid], cmap="gray")
    ax.set_title(f"{name} (z={z_mid})")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Per-channel preprocessing, segmentation, and quantification

 Each channel is preprocessed and segmented in a separate sub-section, allowing to save and visualise intermediate steps for quality checks.

We need to define:

1. Number of images to test (num_test_volumes): Defines how many test images from the input_folder will be used for the trial analysis throughout
2. Downsampling (downsize_factor): Reduces memory usage and speeds up computation.
3. Preprocessing: Gaussian filtering and normalization (either per-slice or across the full 3D stack).

In [ ]:
# --------------------
# INPUT SETTINGS
# --------------------

### How many images to segment? More takes longer but gives more info
num_test_volumes = 1

### downsieze factor
downsize_factor = 1         # scaling factor or 1 to keep original                            
per_slice_norm = True       # True = normalize per-slice, False = normalize whole stack

##Gaussian filter

sigma_um = None             # desired Gaussian sigma in micrometers

per_slice=False             #if True normalize each slice separately

resize_isotropic: False     #if True resize to isotropic voxel size based on voxel_size_um


ram_limit_bytes: int = 2_000_000_000 #memory warning threshold for gaussian filter

order = 1   #Interpolation order for resizing (1 = bilinear, 3 = bicubic, etc.).


In [ ]:
# import tifffile
# import glob
# import numpy as np
# import os
# from skimage.transform import resize

# # we are going to load the images, a sequence of tiff, as a 3D numpy array 
# # and return the names

# tif_files = sorted(glob.glob(os.path.join(folder, "*.tif")))
# if not tif_files:
#     raise FileNotFoundError(f"No TIFF files found in folder: {folder}")
    
# print(f"Found {len(tif_files)} TIFF files.")
    
# frames_in = np.stack([tifffile.imread(f) for f in tif_files], axis=0)
    
# if testing:
#     print(f"Downsizing frames to {downsize_size} for testing...")
#     frames = np.zeros((frames_in.shape[0], downsize_size[0], downsize_size[1]), dtype=frames_in.dtype)
#     for i, frame in enumerate(frames_in):
#         frames[i] = resize(frame, downsize_size, preserve_range=True).astype(frame.dtype)
# else:
#     frames = frames_in
    
# file_names = [os.path.basename(f) for f in tif_files]


a) **Nucleus**

This section describes the preprocessing, segmentation, and quantification of nuclei in 3D microscopy images. The workflow uses optionally either StarDist or Cellpose models for segmentation, with post-processing and quality control steps available.

1. **Preprocessing**: normalize nucleus channel, apply Gaussian filter to reduce noise, optionally downsample.

2. **Segmentation**: using DL models:

    - **StarDist** (probability-based segmentation, very light, works best with more round-shape nuclear objects): controlled by probability (prob_thresh) and overlap (nms_thresh) thresholds.

    - **Cellpose** (SAM-based segmentation, preferrably should be run on GPU)

    Output: labeled 3D mask where each nucleus is assigned a unique label.

3. **Quality Control (QC)**
Save .tif mask stack and .png overlays (maximum-intensity projections with labels).
View some masks in the cell output to quickly check performance.

In [ ]:
import numpy as np
import warnings
from skimage.exposure import rescale_intensity
from skimage.transform import resize
from scipy.ndimage import gaussian_filter as gf, sobel

# we are going to work with an image for the demo, for example the first one

img_3d = all_volumes[0]

img_3d = img_3d.astype('float32')

if per_slice:
    img_3d = np.empty_like(img_3d, dtype='float32')
    for z in range(img_3d.shape[0]):
        img_3d[z] = rescale_intensity(img_3d[z], out_range=(0, 1))
else:
    img_3d = rescale_intensity(img_3d, out_range=(0, 1))

# Check if all slices have the same shape
shapes = [img.shape for img in img_3d]
if len(set(shapes)) > 1:
    warnings.warn("Images in stack have different sizes.")




#apply a gaussian filter
    # ---- pre-checks ----
assert img.ndim == 3 and np.issubdtype(img.dtype, np.number), "Input must be 3D numeric."
assert voxel_size_um is not None, "Voxel size must be provided."
assert len(voxel_size_um) == 3 and all(v > 0 for v in voxel_size_um), "Invalid voxel size."
assert len(sigma_um) == 3 and all(np.isfinite(s) and s > 0 for s in sigma_um), "Invalid sigma."

# convert sigma from µm to voxel units
if sigma_um is None:
    # print warning and set default
    print("[WARN] sigma_um not provided, using default (1, 1, 1) µm.")
    sigma_um = (1.0, 1.0, 1.0)

# convert sigma from µm to voxel units
# for each value, divide desired sigma by actual voxel size
sigma_vox = tuple(s / v for s, v in zip(sigma_um, voxel_size_um))

# Dynamic range check
if np.all(img == img.flat[0]):
    print("[WARN] Input is constant intensity.")

# Memory estimate
est_bytes = img.size * img.itemsize * 5
if est_bytes > ram_limit_bytes:
    print(f"[WARN] Estimated RAM {est_bytes/1e9:.2f} GB exceeds limit.")

# ---- filtering ----
out = gf(img, sigma=sigma_vox, mode="reflect")

# ---- post-checks ----
if out.shape != img.shape:
    print("[FAIL] Shape changed unexpectedly.")

# mad is a robust noise estimator
# calculated by median absolute deviation from median
mad_before = np.median(np.abs(img - np.median(img)))
mad_after = np.median(np.abs(out - np.median(out)))

# edge preservation check
# use Sobel filter to estimate edges and make sure they are similar before and after
# we calculate the mean gradient magnitude across the whole volume
grad_before = np.mean(np.sqrt(sum(sobel(img, axis=i)**2 for i in range(3))))
grad_after = np.mean(np.sqrt(sum(sobel(out, axis=i)**2 for i in range(3))))
# we want the edges to be at least half as strong after blurring
if grad_after < 0.5 * grad_before:
    print("[WARN] Edges weakened too much (over-smoothing).")

# print QC, including sigma in voxels, noise drop percentage, and edge ratio
print(f"MAD before: {mad_before:.4g}, MAD after: {mad_after:.4g}")
print(f"Edge ratio (mean gradident before / mean gradient after) {grad_after/grad_before:.2f}")



# the resolution in z axis is always worse (PSF) so we want our 3D data to be isometric, for that
# we reshape the data

if img_3d.ndim != 3:
    raise ValueError("Input must be a 3D array (Z, Y, X).")

if len(voxel_size_um) != 3 or any(s <= 0 for s in voxel_size_um):
    raise ValueError("voxel_size_um must be a tuple of 3 positive floats (sz, sy, sx).")

sz, sy, sx = voxel_size_um
min_voxel = min(voxel_size_um)
scale_factors = np.array([sz, sy, sx]) / min_voxel

new_shape = np.round(np.array(img_3d.shape) * scale_factors).astype(int)

img_iso = resize(
    img_3d,
    new_shape,
    order=order,
    anti_aliasing=True,
    preserve_range=True
).astype(img_3d.dtype)


# Data size makes the analysis slower, sometimes is worthy to downsample the data 


z, y, x = img_3d.shape
new_y = int(round(y * downsize_factor))
new_x = int(round(x * downsize_factor))

img_resized = resize(
    img_3d,
    (z, new_y, new_x),
    order=order,
    anti_aliasing=True,
    preserve_range=True
).astype(img_3d.dtype)

    

In [ ]:
# 2. nucleus segmentation

##### params #####
gaussian_nucleus = config["preprocessing"]["gaussian_nucleus"]  # apply gaussian filter to nuclei channel
sigma_um_nucleus = [float(v) for v in config["preprocessing"]["gaussian_sigma_nucleus"] ] # value of sigma for gaussian - could be adjusted depending on the data
use_model = config["segmentation"]["use_model"]  # "cellpose" or "stardist"

prob_thresh = config["segmentation"]["prob_thresh"]  # stardist param: sets the minimum confidence for accepting a predicted nucleus
nms_thresh = config["segmentation"]["nms_thresh"]    # stardist param: remove detections overlapping by more than this threshold

gpu= config["segmentation"]["gpu"]           # cellpose param: running without GPU is very slow

In [ ]:
##################

results_dict = {} # to save the results

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name
    
    # --- 2.1 nuclei preprocessing ---
    nuclei_volume = v["channels"]["nucleus"]
    nuclei_norm = preprocess_3d_image(nuclei_volume, 
                                      downsize_factor=downsize_factor,
                                      apply_gaussian_filter=gaussian_nucleus,           
                                      voxel_size_um=voxel_size_um,
                                      sigma_um=sigma_um_nucleus)             # could be adjusted depending on the data
    
    # --- 2.2 nuclei segmentation ---
    if use_model == "stardist":
        nuclei_mask = segment_with_stardist(nuclei_norm, prob_thresh=prob_thresh, nms_thresh=nms_thresh)
    elif use_model == "cellpose":
        nuclei_mask = segment_with_cellpose(nuclei_norm, gpu=gpu)
    
    # saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist

    results_dict[file_name]["nuclei_mask"] = nuclei_mask 
    
    # 2.3 save nuclei mask (.tiff) as well as PNG overlay image for quality check
    save_segmentation_results(
        nuclei_norm, 
        nuclei_mask, 
        output_root=output_folder, 
        experiment_label=f"{file_name}_nuclei",
        save_overlay=True
    )

In [ ]:
### View some masks - are they sensible?
# up to 5 pairs of (index, fname)
pairs = list(enumerate(results_dict.keys()))[:5]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 5))
if len(pairs) == 1:
    axes = [axes]

for ax, (idx, fname) in zip(axes, pairs):
    vol = all_volumes[idx]["channels"]
    mask = results_dict[fname]["nuclei_mask"]

    if len(vol.keys()) == 3:
        mid = len(vol["nucleus"]) // 2
        img = vol["nucleus"][mid]
        mask_slice = mask[mid]
    else:
        img = vol
        mask_slice = mask

    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(mask_slice == 0, mask_slice),
              cmap="autumn", alpha=0.5)
    ax.set_title(fname)
    ax.axis("off")

plt.tight_layout()
plt.show()

4. **Quantification** (optional)

- Extract per-object features such as count and volume.

- Compare measured nucleus volumes against expected biological range (e.g. 10 µm diameter sphere).

In [ ]:
# 2.4 (optional) quantify nuclei

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name
    
    # --- retrieve nuclei masks ---
    nuclei_mask = results_dict[file_name]["nuclei_mask"]
    nuclei_mask = tidy_mask(nuclei_mask)
    
    # --- quantify nuclei ---
    df, summary = quantify_objects(nuclei_mask, voxel_size=voxel_size_um, features=("count","volume"))
    print(summary)

In [ ]:
# 2.4 example thresholds:
expected_d_um_nuclei = config["expected_d_um_nuclei"]     # e.g., 10 µm nuclei
tolerance_size_nuclei = config["tolerance_size_nuclei"]                     # ±25% tolerance

min_um3 = sphere_volume_um3(expected_d_um_nuclei * (1 - tolerance_size_nuclei))
max_um3 = sphere_volume_um3(expected_d_um_nuclei * (1 + tolerance_size_nuclei))
print("Expected nuclei volume:", min_um3, "to", max_um3, "µm3")

5. **Filtering** (optional)

- Remove objects outside expected size tolerance.

- Update results dictionary with filtered masks.

In [ ]:
# 2.5 (optional) filtering out nuclei with unwanted size
filter_nuclei = config["filter_nuclei"]     # True of False

if filter_nuclei:
    for v in all_volumes[:num_test_volumes]:
        file_name = v['filename']  # preserve original file name

        # step 1: get mask
        nuclei_mask = results_dict[file_name]["nuclei_mask"]
        nuclei_mask = tidy_mask(nuclei_mask)

        # step 2: filter mask
        nuclei_mask_filtered, df = filter_mask_by_size(nuclei_mask, expected_d_um_nuclei, tolerance_size_nuclei, voxel_size=voxel_size_um, return_df=True)
        
        n_before = len(df)
        n_after = df["keep"].sum() if len(df) > 0 else 0
        
        # step 3: replace mask in results dict for downstream use
        results_dict[file_name]["nuclei_mask"] = nuclei_mask_filtered

        print(f'{file_name}:',"Before filtering:", n_before)
        
        print("After filtering:", n_after)


b) ** Cytoplasm **

1. Preprocessing: normalise cytoplasm channel, Gaussian filtering typically disabled in "intensity" mode

2. Segmentation

- Cytoplasm segmented via watershed, seeded by nucleus masks.

- Two strategies available:
    - "intensity" (default, based on cytoplasmic intensity)
    - "membrane" (alternative, based on membrane-labeled channels)

3. Saving Results: store cytoplasm mask in results dictionary, save .tif mask and .png overlay for QC

In [ ]:
# ---------------------------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------------------------



# cytoplasm
gaussian_cytoplasm = False      # apply gaussian blue to cytoplasm channel
gaussian_sigma_cytoplasm =  [1,1,1]   # Standard deviation for Gaussian smoothing (µm units)

mode = 'membrane'   # chose 'intensity' if you don't have a membrane marker
# organelle
gaussian_organelle = False      # apply gaussian blue to organelle channel
gaussian_sigma_organelle = [1,1,1]   # Standard deviation for Gaussian smoothing (µm units)

background_subtract = False # true/false: whether to apply background subtraction
normalize = False            # true/false: intensity normalization


In [ ]:
# --- 3.1 cytoplasm channel preprocessing ---
# we need to call the cytoplam channel that we map before
# and assign a variable to it. 
# Then we can choose to dowsampling, rescale intensity,
# make it isometric and apply a gaussian filter 
# before segmenting

# ---------------------------------------------------------------------------
# Segmentation of the membrane
# ---------------------------------------------------------------------------

cyto_channel = all_volumes[2]       # is this a cytoplasmic 3D array?
membrane_threshold = 0.2        # Threshold for membrane signal to act as stopping boundary.
min_signal = 0.05       # Minimum normalized intensity for cytoplasm mask (for intensity mode).

# Downsampling

z, y, x = cyto_channel.shape
new_y = int(round(y * downsize_factor))
new_x = int(round(x * downsize_factor))

cyto_channel_resized = resize(
    img_3d,
    (z, new_y, new_x),
    order=order,
    anti_aliasing=True,
    preserve_range=True
).astype(img_3d.dtype)

# rescale intensity




# Gaussian to cytoplams


# isotropic

sz, sy, sx = voxel_size_um
min_voxel = min(voxel_size_um)
scale_factors = np.array([sz, sy, sx]) / min_voxel

new_shape = np.round(np.array(img_3d.shape) * scale_factors).astype(int)

cyto_channel_iso = resize(
    cyto_channel,                           # check the variable
    new_shape,
    order=order,
    anti_aliasing=True,
    preserve_range=True
).astype(img_3d.dtype)

# --- 3.2 cytoplasm segmentation (watershed) ---
# Using a cytoplasmic protein or a membrane marker

from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.filters import gaussian

# Segment cytoplasm using nuclei as seeds and cytoplasmic/membrane channel as guidance.
    
if mode == "membrane":
    # binary mask of membrane
    membrane_mask = cyto_channel > membrane_threshold
    distance = ndi.distance_transform_edt(~membrane_mask)
    cytoplasm_labels = watershed(-distance, markers=nuclei_mask, mask=~membrane_mask)

elif mode == "intensity":
    # smooth the channel for better segmentation
    smoothed = gaussian(cyto_channel, sigma=gaussian_sigma_cytoplasm) ### pre-processing should be done in the separate step prior to segmentation
    #smoothed = cyto_channel 

    # cytoplasmic regions to include
    cytoplasm_mask = smoothed > min_signal
    inverted = -smoothed
    cytoplasm_labels = watershed(inverted, markers=nuclei_mask, mask=cytoplasm_mask)

else:
    raise ValueError("Invalid mode. Choose 'membrane' or 'intensity'.")

In [ ]:
### View some masks - are they sensible?
# up to 5 pairs of (index, fname)
pairs = list(enumerate(results_dict.keys()))[:5]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 5))
if len(pairs) == 1:
    axes = [axes]

for ax, (idx, fname) in zip(axes, pairs):
    vol = all_volumes[idx]["channels"]
    mask = results_dict[fname]["cytoplasm_mask"]

    if len(vol.keys()) == 3:
        mid = len(vol["cytoplasm"]) // 2
        img = vol["cytoplasm"][mid]
        mask_slice = mask[mid]
    else:
        img = vol
        mask_slice = mask

    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(mask_slice == 0, mask_slice),
              cmap="autumn", alpha=0.5)
    ax.set_title(fname)
    ax.axis("off")

plt.tight_layout()
plt.show()

---
#### c) organelle / intracellular structure

This sub-section deals with segmenting/quantifying intracellular structures per cell mask, mapping the mask from intracellular channel to the cell_ID label from cytoplasmic mask. The general overview of the process is as follows: 

4.1 **Preprocessing**
- 4.0: obtain the suggested parameters for the preprocessing (from `suggest_normalization_param` helper function)
- Apply recommended normalization parameters (from `suggest_normalization_param`).  
  - Example: scaling intensity to `[0, 17]`.  
  - (optionally) downsample to match cytoplasm resolution -- required for aligning shapes of different channels  
- Apply Gaussian smoothing (e.g. σ = 1).  

4.2 **Segmentation**
- Use **AllenCell wrappers** (e.g. `dot_2d_slice_by_slice_wrapper` for spotty structures), based on the **lookup tables** found here:
  - General pipelines: https://www.allencell.org/segmenter.html#lookup-table 
  - Examples of notebooks with specific functions: https://github.com/AllenCell/aics-segmentation/tree/main/lookup_table_demo
  - Description of all modules available: https://allencell.github.io/aics-segmentation/aicssegmentation.core.html#   
- Returns binary masks (optionally cleaned by removing very small objects).  
- Save `.tif` masks and `.png` overlays for QC.  

4.3 **Quantification**
- Map segmented intracellular structures to corresponding cytoplasm masks.  
- Compute per-object and/or per-cell features:  
  - **count** (number of spots per cell)  
  - **volume** (µm³)  
  - **intensity** (mean intensity per object/cell)  
- Save results into CSV files for downstream analysis.